# Redshift-Space Distortion (RSD) Sub-volume Corrections

In standard clustering, Subvolume subsampling severely artificialises the 1-halo spatial term. 
When working with Redshift-Space Distortions (RSD), peculiar velocities drastically stretch this 1-halo clustering along the line of sight (producing the classic "Fingers-of-God" effect). 

This notebook validates the `compute_weighted_rsd_multipoles` algorithm implemented in the module `galform_analysis.analysis.redshift_space_distortions.subvol_weighted_multipoles`.
We apply sub-vol weighting to isolate the expected Monopole ($\xi_0$) and Quadrupole ($\xi_2$), completely bypassing the substantial anisotropic smearing bias artificially introduced by subsampling!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add the GALFORM python source package
sys.path.append(str(Path("../../src").resolve()))

from analysis.redshift_space_distortions.subvol_weighted_multipoles import (
    compute_weighted_rsd_multipoles,
)
from utils.read_galaxies import read_galaxy_arrays
from utils import setconfig
from config import Cosmology, get_snapshot_redshift

# Apply project plotting style, then small notebook-specific tweaks
setconfig()
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Load Real GALFORM Galaxies (Lightweight Notebook Sample)
This section uses real GALFORM outputs (the same source tree used by the SLURM jobs), but only a small subset so it runs interactively.

We build redshift-space positions with:
- box position: $z$
- line-of-sight velocity: $v_z$ (`vzgal`)
- shift: $\Delta s_\parallel \approx v_z \; h / H(z)$ in $h^{-1}\,\mathrm{Mpc}$

Then we compare RSD multipoles for using 1 vs 2 vs 3 subvolumes.

In [ ]:
# Data location (same family as SLURM workflows)
BASE_DIR = Path("/cosma5/data/durham/dc-hick2/Galform_Out/L800/lc16")
IZ = 155
BOXSIZE = 542.16
K_TOTAL = 1024

# Lightweight notebook settings
MAX_GAL_PER_SUBVOL = 3000
MHALO_MIN = 1e10
CENTRALS_ONLY = False
IVOLS_FOR_COMPARISON = [0, 1, 2]  # compare m=1,2,3 by taking first m ivols

# Convert z-position to redshift space with vzgal
z_snap = get_snapshot_redshift(f"iz{IZ}", 'L800')
if z_snap is None:
    z_snap = 0.0

h = Cosmology.h
Ez = np.sqrt(Cosmology.OMEGA_M * (1.0 + z_snap) ** 3 + Cosmology.OMEGA_L)
H_z = 100.0 * h * Ez  # km/s/Mpc

print(f"Using {BASE_DIR}/iz{IZ}")
print(f"Snapshot redshift z={z_snap:.3f}, H(z)={H_z:.2f} km/s/Mpc")


def load_rsd_positions_for_ivols(ivols):
    pos_chunks = []
    label_chunks = []

    for label, ivol in enumerate(ivols):
        arrays, _ = read_galaxy_arrays(
            iz_path=str(BASE_DIR / f"iz{IZ}"),
            ivol=int(ivol),
            fields=["vzgal"],
            include_positions=True,
            include_derived=True,
            centrals_only=CENTRALS_ONLY,
            mhalo_min=MHALO_MIN,
        )

        x = arrays["x"]
        y = arrays["y"]
        z = arrays["z"]
        vz = arrays["vzgal"]

        # Randomly thin to keep the notebook fast and interactive
        n = len(x)
        if n > MAX_GAL_PER_SUBVOL:
            idx = np.random.choice(n, size=MAX_GAL_PER_SUBVOL, replace=False)
            x, y, z, vz = x[idx], y[idx], z[idx], vz[idx]

        ds_par = (vz / H_z) * h * (1.0 + z_snap)  # Mpc/h
        z_rsd = (z + ds_par) % BOXSIZE

        pos = np.column_stack([x, y, z_rsd])
        labels = np.full(pos.shape[0], label, dtype=np.int64)

        pos_chunks.append(pos)
        label_chunks.append(labels)

        print(f"ivol{ivol}: loaded {pos.shape[0]} galaxies")

    gal_pos = np.vstack(pos_chunks)
    gal_labels = np.concatenate(label_chunks)
    return gal_pos, gal_labels

## 2. Compute RSD Multipoles for 1, 2, and 3 Subvolumes
For each $m \in \{1,2,3\}$, we concatenate the first $m$ subvolumes, compute standard and corrected $\xi(s,\mu)$, and project to:
- monopole $\xi_0(s)$
- quadrupole $\xi_2(s)$

In [ ]:
# Corrfunc binning (kept moderate for notebook runtime)
s_bins = np.logspace(-1, 1.4, 18)
mu_max = 1.0
n_mu_bins = 16

results_by_m = {}

for m in [1, 2, 3]:
    ivols = IVOLS_FOR_COMPARISON[:m]
    gal_pos, labels = load_rsd_positions_for_ivols(ivols)

    # Use a moderate random catalogue size for speed
    n_random = max(12000, int(2.5 * gal_pos.shape[0]))
    rand_pos = np.random.uniform(0.0, BOXSIZE, size=(n_random, 3))

    res = compute_weighted_rsd_multipoles(
        galaxy_pos=gal_pos,
        galaxy_labels=labels,
        random_pos=rand_pos,
        s_bins=s_bins,
        mu_max=mu_max,
        n_mu_bins=n_mu_bins,
        k_total=K_TOTAL,
        boxsize=BOXSIZE,
        nthreads=4,
    )

    results_by_m[m] = res
    print(
        f"m={m}: ngal={gal_pos.shape[0]}, nrand={n_random}, "
        f"alpha={res['alpha']:.6f}, beta={res['beta'] if np.isfinite(res['beta']) else np.nan}"
    )

## 3. RSD Curves + Differences By Subvolume Count (All CSV Sets)
This section discovers all available subvolume RSD CSV outputs and processes each dataset group:
- model, simulation, snapshot, and selection-tag (unseeded vs seeded)

For each group, we generate three figure sets with the same layout (standard left, corrected right; monopole top, quadrupole bottom):
- raw $s^2\xi_\ell(s)$ curves with full-box/reference overlays
- absolute difference from the full-box/reference line
- percentage difference from the full-box/reference line

Reference selection:
- if a dedicated full-box standard CSV exists, use it for the standard panel
- otherwise use the largest available $n_{subvol}$ as the proxy reference
- for corrected panels, use $n_{subvol}=1024$ when present, otherwise the largest available $n_{subvol}$

In [ ]:
import re
from collections import defaultdict
from IPython.display import display

DATA_ROOT = Path("../../data/redshift_space_distortions/subvol_jobs")
PLOT_ROOT = Path("./_plots/rsd_weighted_correction")
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

SUBVOL_FILE_PATTERN = re.compile(
    r"^rsd_subvol_(normal|corrected)_(?P<sim>[^_]+)_(?P<model>[^_]+)_iz(?P<iz>\d+)_n(?P<n_subvol>\d+)(?P<seeded>_ivselrandom)?\.csv$"
)

# None means display all discovered groups in notebook output.
# To limit displayed groups later, set this to a list of tuples:
# [("lc16", "L800", 155, "unseeded"), ("lc16", "L800", 207, "unseeded")]
SHOW_GROUPS_IN_NOTEBOOK = None


def parse_subvol_metadata(csv_path):
    m = SUBVOL_FILE_PATTERN.match(csv_path.name)
    if m is None:
        return None

    mode = m.group(1)
    return {
        "path": csv_path,
        "mode": mode,
        "sim": m.group("sim"),
        "model": m.group("model"),
        "iz": int(m.group("iz")),
        "n_subvol": int(m.group("n_subvol")),
        "set_tag": "seeded" if m.group("seeded") else "unseeded",
    }


def discover_rsd_records(data_root):
    rows = []
    for csv_path in sorted(data_root.rglob("*.csv")):
        meta = parse_subvol_metadata(csv_path)
        if meta is not None:
            rows.append(meta)

    records = pd.DataFrame(rows)
    if records.empty:
        raise RuntimeError(f"No subvolume RSD CSV files found under: {data_root}")

    records["group_key"] = records.apply(
        lambda row: f"{row['model']}|{row['sim']}|iz{row['iz']}|{row['set_tag']}",
        axis=1,
    )
    return records


def load_mode_tables(group_df):
    tables = {"normal": {}, "corrected": {}}
    for _, row in group_df.iterrows():
        df = pd.read_csv(row["path"]).sort_values("s")
        tables[row["mode"]][int(row["n_subvol"])] = df
    return tables


def get_standard_fullbox_path(model, sim, iz):
    return DATA_ROOT / model / sim / f"iz{iz}" / "normal" / f"rsd_fullbox_standard_{sim}_{model}_iz{iz}_n1024.csv"


def choose_reference(group_id, mode, mode_tables):
    model, sim, iz, set_tag = group_id

    if not mode_tables:
        return None, None, None

    if mode == "normal" and set_tag == "unseeded":
        fullbox_path = get_standard_fullbox_path(model, sim, iz)
        if fullbox_path.exists():
            return pd.read_csv(fullbox_path).sort_values("s"), "full-box standard", 1024

    if 1024 in mode_tables:
        return mode_tables[1024].sort_values("s"), "n=1024 proxy full-box", 1024

    n_ref = max(mode_tables)
    return mode_tables[n_ref].sort_values("s"), f"n={n_ref} proxy full-box", n_ref


def aligned_s2_multipole(run_df, ref_df, multipole):
    merged = pd.merge(
        run_df[["s", multipole]],
        ref_df[["s", multipole]],
        on="s",
        how="inner",
        suffixes=("_run", "_ref"),
    )
    if merged.empty:
        return None, None, None

    s = merged["s"].to_numpy()
    run = (merged["s"] ** 2 * merged[f"{multipole}_run"]).to_numpy()
    ref = (merged["s"] ** 2 * merged[f"{multipole}_ref"]).to_numpy()
    return s, run, ref


def safe_plot_stem(model, sim, iz, set_tag):
    return f"{model}_{sim}_iz{iz}_{set_tag}".replace("/", "-").replace(" ", "_")


records_df = discover_rsd_records(DATA_ROOT)
display(
    records_df[["model", "sim", "iz", "set_tag", "mode", "n_subvol", "path"]]
    .sort_values(["model", "sim", "iz", "set_tag", "mode", "n_subvol"])
    .head(40)
)

plot_outputs = []
summary_rows = []
displayed_group_count = 0

for group_id, group_df in records_df.groupby(["model", "sim", "iz", "set_tag"], dropna=False):
    model, sim, iz, set_tag = group_id
    tables = load_mode_tables(group_df)

    normal_ref_df, normal_ref_label, normal_ref_n = choose_reference(group_id, "normal", tables["normal"])
    corrected_ref_df, corrected_ref_label, corrected_ref_n = choose_reference(group_id, "corrected", tables["corrected"])

    summary_rows.append(
        {
            "model": model,
            "sim": sim,
            "iz": iz,
            "set_tag": set_tag,
            "n_normal": len(tables["normal"]),
            "n_corrected": len(tables["corrected"]),
            "normal_ref": normal_ref_label,
            "corrected_ref": corrected_ref_label,
        }
    )

    n_all = sorted(set(list(tables["normal"].keys()) + list(tables["corrected"].keys())))
    if not n_all:
        continue
    color_map = {
        n_sub: color
        for n_sub, color in zip(n_all, plt.cm.viridis(np.linspace(0.1, 0.95, len(n_all))))
    }

    figs = []

    # Figure A: raw curves
    fig_raw, axes_raw = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
    figs.append(("curves", fig_raw, axes_raw))

    # Figure B: absolute differences to reference
    fig_abs, axes_abs = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
    figs.append(("absdiff", fig_abs, axes_abs))

    # Figure C: percentage differences to reference
    fig_pct, axes_pct = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
    figs.append(("pctdiff", fig_pct, axes_pct))

    mode_layout = [
        ("normal", tables["normal"], normal_ref_df, normal_ref_label, normal_ref_n, 0),
        ("corrected", tables["corrected"], corrected_ref_df, corrected_ref_label, corrected_ref_n, 1),
    ]

    eps = 1e-12

    for mode, mode_tables, ref_df, ref_label, ref_n, col in mode_layout:
        ax_raw_0 = axes_raw[0, col]
        ax_raw_2 = axes_raw[1, col]

        ax_abs_0 = axes_abs[0, col]
        ax_abs_2 = axes_abs[1, col]

        ax_pct_0 = axes_pct[0, col]
        ax_pct_2 = axes_pct[1, col]

        if ref_df is None or not mode_tables:
            for ax in [ax_raw_0, ax_raw_2, ax_abs_0, ax_abs_2, ax_pct_0, ax_pct_2]:
                ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center", va="center")
                ax.set_axis_off()
            continue

        for n_sub in sorted(mode_tables):
            run_df = mode_tables[n_sub]
            color = color_map[n_sub]
            label = f"n={n_sub}"

            # xi0
            s0, run0, ref0 = aligned_s2_multipole(run_df, ref_df, "xi0")
            if s0 is not None:
                ax_raw_0.plot(s0, run0, lw=2, color=color, alpha=0.9, label=label)
                ax_abs_0.plot(s0, np.abs(run0 - ref0), lw=2, color=color, alpha=0.9, label=label)
                pct0 = 100.0 * (run0 - ref0) / (np.abs(ref0) + eps)
                ax_pct_0.plot(s0, pct0, lw=2, color=color, alpha=0.9, label=label)

            # xi2
            s2, run2, ref2 = aligned_s2_multipole(run_df, ref_df, "xi2")
            if s2 is not None:
                ax_raw_2.plot(s2, run2, lw=2, color=color, alpha=0.9, label=label)
                ax_abs_2.plot(s2, np.abs(run2 - ref2), lw=2, color=color, alpha=0.9, label=label)
                pct2 = 100.0 * (run2 - ref2) / (np.abs(ref2) + eps)
                ax_pct_2.plot(s2, pct2, lw=2, color=color, alpha=0.9, label=label)

        # Overlay reference curves
        ax_raw_0.plot(
            ref_df["s"].to_numpy(),
            (ref_df["s"] ** 2 * ref_df["xi0"]).to_numpy(),
            color="black",
            lw=3,
            linestyle="--",
            label=ref_label,
        )
        ax_raw_2.plot(
            ref_df["s"].to_numpy(),
            (ref_df["s"] ** 2 * ref_df["xi2"]).to_numpy(),
            color="black",
            lw=3,
            linestyle="--",
            label=ref_label,
        )

        # Axis formatting for this mode column
        for ax in [ax_raw_0, ax_raw_2, ax_abs_0, ax_abs_2, ax_pct_0, ax_pct_2]:
            ax.set_xscale("log")
            ax.grid(True, alpha=0.3)

        ax_raw_0.set_yscale("log")
        ax_raw_2.set_yscale("symlog", linthresh=10.0)

        ax_raw_0.set_title(f"{mode.title()} monopole ({ref_label})")
        ax_raw_2.set_title(f"{mode.title()} quadrupole ({ref_label})")
        ax_abs_0.set_title(f"{mode.title()} monopole |run - ref|")
        ax_abs_2.set_title(f"{mode.title()} quadrupole |run - ref|")
        ax_pct_0.set_title(f"{mode.title()} monopole % diff")
        ax_pct_2.set_title(f"{mode.title()} quadrupole % diff")

        ax_pct_0.axhline(0.0, color="black", lw=1.5, linestyle="--")
        ax_pct_2.axhline(0.0, color="black", lw=1.5, linestyle="--")

    # Shared labels and legends
    axes_raw[0, 0].set_ylabel(r"$s^2\,\xi_0(s)$")
    axes_raw[1, 0].set_ylabel(r"$s^2\,\xi_2(s)$")
    axes_raw[1, 0].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")
    axes_raw[1, 1].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")

    axes_abs[0, 0].set_ylabel(r"$|\Delta(s^2\xi_0)|$")
    axes_abs[1, 0].set_ylabel(r"$|\Delta(s^2\xi_2)|$")
    axes_abs[1, 0].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")
    axes_abs[1, 1].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")

    axes_pct[0, 0].set_ylabel(r"$100\,\Delta/(|ref|+\epsilon)$")
    axes_pct[1, 0].set_ylabel(r"$100\,\Delta/(|ref|+\epsilon)$")
    axes_pct[1, 0].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")
    axes_pct[1, 1].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")

    axes_raw[0, 1].legend(fontsize=8, ncol=2)
    axes_raw[1, 1].legend(fontsize=8, ncol=2)
    axes_abs[0, 1].legend(fontsize=8, ncol=2)
    axes_abs[1, 1].legend(fontsize=8, ncol=2)
    axes_pct[0, 1].legend(fontsize=8, ncol=2)
    axes_pct[1, 1].legend(fontsize=8, ncol=2)

    group_title = f"{model} | {sim} | iz{iz} | {set_tag}"
    fig_raw.suptitle(f"RSD curves by subvolume: {group_title}", y=1.01)
    fig_abs.suptitle(f"RSD absolute difference to reference: {group_title}", y=1.01)
    fig_pct.suptitle(f"RSD percentage difference to reference: {group_title}", y=1.01)

    for _, fig, _ in figs:
        fig.tight_layout()

    stem = safe_plot_stem(model, sim, iz, set_tag)
    for tag, fig, _ in figs:
        outpath = PLOT_ROOT / f"rsd_{stem}_{tag}.png"
        fig.savefig(outpath, bbox_inches="tight")
        plot_outputs.append(outpath)

    show_group = (SHOW_GROUPS_IN_NOTEBOOK is None) or (group_id in SHOW_GROUPS_IN_NOTEBOOK)
    if show_group:
        for _, fig, _ in figs:
            display(fig)
        displayed_group_count += 1

    for _, fig, _ in figs:
        plt.close(fig)

summary_df = pd.DataFrame(summary_rows).sort_values(["model", "sim", "iz", "set_tag"])
print(f"Discovered {len(records_df)} subvolume CSV files across {summary_df.shape[0]} dataset groups.")
print(f"Saved {len(plot_outputs)} figure files to {PLOT_ROOT}")
print(f"Displayed plots for {displayed_group_count} dataset groups in notebook output.")

display(summary_df)

## 4. Convergence Tables Across All RSD CSV Sets
This section computes convergence metrics for every discovered dataset group and both estimator modes.

For each $n_{subvol}$ relative to its mode-specific reference, we report:
- median absolute difference in $s^2\xi_0(s)$
- median absolute difference in $s^2\xi_2(s)$
- median absolute percentage difference in $s^2\xi_0(s)$
- median absolute percentage difference in $s^2\xi_2(s)$

This gives a compact numerical view of convergence quality across all available RSD CSV outputs.

In [ ]:
eps = 1e-12
convergence_rows = []

for group_id, group_df in records_df.groupby(["model", "sim", "iz", "set_tag"], dropna=False):
    model, sim, iz, set_tag = group_id
    tables = load_mode_tables(group_df)

    for mode in ["normal", "corrected"]:
        mode_tables = tables[mode]
        ref_df, ref_label, ref_n = choose_reference(group_id, mode, mode_tables)
        if ref_df is None:
            continue

        for n_subvol, run_df in sorted(mode_tables.items()):
            # Monopole
            s0, run0, ref0 = aligned_s2_multipole(run_df, ref_df, "xi0")
            # Quadrupole
            s2, run2, ref2 = aligned_s2_multipole(run_df, ref_df, "xi2")
            if s0 is None or s2 is None:
                continue

            abs_diff0 = np.abs(run0 - ref0)
            abs_diff2 = np.abs(run2 - ref2)

            pct_diff0 = 100.0 * (run0 - ref0) / (np.abs(ref0) + eps)
            pct_diff2 = 100.0 * (run2 - ref2) / (np.abs(ref2) + eps)

            convergence_rows.append(
                {
                    "model": model,
                    "sim": sim,
                    "iz": iz,
                    "set_tag": set_tag,
                    "mode": mode,
                    "n_subvol": int(n_subvol),
                    "reference": ref_label,
                    "reference_n": ref_n,
                    "med_abs_diff_s2xi0": float(np.nanmedian(abs_diff0)),
                    "med_abs_diff_s2xi2": float(np.nanmedian(abs_diff2)),
                    "med_abs_pct_diff_s2xi0": float(np.nanmedian(np.abs(pct_diff0))),
                    "med_abs_pct_diff_s2xi2": float(np.nanmedian(np.abs(pct_diff2))),
                }
            )

convergence_df = pd.DataFrame(convergence_rows).sort_values(
    ["model", "sim", "iz", "set_tag", "mode", "n_subvol"]
)

print("Convergence metrics (first 30 rows):")
display(convergence_df.head(30))

summary_by_group = (
    convergence_df.groupby(["model", "sim", "iz", "set_tag", "mode"], dropna=False)
    .agg(
        n_curves=("n_subvol", "nunique"),
        best_med_abs_pct_s2xi0=("med_abs_pct_diff_s2xi0", "min"),
        best_med_abs_pct_s2xi2=("med_abs_pct_diff_s2xi2", "min"),
        worst_med_abs_pct_s2xi0=("med_abs_pct_diff_s2xi0", "max"),
        worst_med_abs_pct_s2xi2=("med_abs_pct_diff_s2xi2", "max"),
    )
    .reset_index()
    .sort_values(["model", "sim", "iz", "set_tag", "mode"])
)

print("Per-group convergence summary:")
display(summary_by_group)

# Save tables for downstream use
conv_path = PLOT_ROOT / "rsd_convergence_metrics_all_sets.csv"
summary_path = PLOT_ROOT / "rsd_convergence_summary_all_sets.csv"
convergence_df.to_csv(conv_path, index=False)
summary_by_group.to_csv(summary_path, index=False)
print(f"Saved convergence metrics table: {conv_path}")
print(f"Saved convergence summary table: {summary_path}")